In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch

from functools import partial
from transformer_lens import HookedTransformer
from sae_lens import SAE

BASE_PRISM_DIR = r"C:\Users\thors\Documents\GitHub\prism"

METRICS_CSV = os.path.join(
    BASE_PRISM_DIR,
    r"results\meta-evaluation_cosine-similarity_target-gpt2-small-sae_textgen-gemini-1-5-pro_mean_evalgen-gemini-1-5-pro_cosmopedia_1000.csv"
)

PROMPTS_CSV = r"C:\Users\thors\Documents\GitHub\llm-interpretability\llm-interpretability\data\gpt_data\gpt-2-layer0.csv"

EVAL_MODEL_NAME = "gpt2-small"
SAE_RELEASE = "gpt2-small-resid-post-v5-32k"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cpu"

DEFAULT_LAYER = 0  

In [ ]:
metrics = pd.read_csv(METRICS_CSV)

metrics = metrics[[
    "layer",
    "unit",
    "cosine_similarity",
    "cosine_similarity_random",
    "max_mad",
]].drop_duplicates(["layer", "unit"]).reset_index(drop=True)

print("PRISM metrics rows:", len(metrics))

In [ ]:
eval_model = HookedTransformer.from_pretrained(
    EVAL_MODEL_NAME,
    device=device,
    dtype=torch.float32,
)
eval_model.eval()

sae_cache = {}

def hook_name(layer: int) -> str:
    return f"blocks.{layer}.hook_resid_post"

def get_sae(layer: int) -> SAE:
    if layer in sae_cache:
        return sae_cache[layer]
    sae_id = f"blocks.{layer}.hook_resid_post"
    sae, cfg, sparsity = SAE.from_pretrained_with_cfg_and_sparsity(
        SAE_RELEASE,
        sae_id,
        device=device,
    )
    sae.eval()
    sae_cache[layer] = sae
    return sae

In [ ]:
prompts = pd.read_csv(PROMPTS_CSV)

prompts["layer"] = DEFAULT_LAYER
prompts["label"] = prompts["label"].astype(str).str.strip().str.lower()
prompts["text"] = prompts["text"].astype(str)

print(prompts["label"].value_counts())
print("rows:", len(prompts))

In [ ]:
def per_sample_nll(logits: torch.Tensor, tokens: torch.Tensor) -> torch.Tensor:
    shift_logits = logits[:, :-1, :]
    shift_labels = tokens[:, 1:]
    logp = torch.log_softmax(shift_logits, dim=-1)
    nll = -logp.gather(dim=-1, index=shift_labels.unsqueeze(-1)).squeeze(-1)
    return nll.mean(dim=1)

@torch.no_grad()
def losses_baseline(texts: list[str]) -> np.ndarray:
    tokens = eval_model.to_tokens(texts).to(device)
    logits = eval_model(tokens, return_type="logits")
    return per_sample_nll(logits, tokens).cpu().numpy()

def hook_recon_only(acts: torch.Tensor, hook, sae: SAE):
    B, S, D = acts.shape
    flat = acts.reshape(-1, D)
    feats = sae.encode(flat)
    recon = sae.decode(feats)
    return recon.reshape(B, S, D)

def hook_ablate_feature(acts: torch.Tensor, hook, sae: SAE, feature_idx: int):
    B, S, D = acts.shape
    flat = acts.reshape(-1, D)
    feats = sae.encode(flat)
    feats[:, int(feature_idx)] = 0.0
    recon = sae.decode(feats)
    return recon.reshape(B, S, D)

@torch.no_grad()
def losses_recon_only(texts: list[str], layer: int) -> np.ndarray:
    sae = get_sae(layer)
    tokens = eval_model.to_tokens(texts).to(device)
    logits = eval_model.run_with_hooks(
        tokens,
        return_type="logits",
        fwd_hooks=[(hook_name(layer), partial(hook_recon_only, sae=sae))],
    )
    return per_sample_nll(logits, tokens).cpu().numpy()

@torch.no_grad()
def losses_ablate(texts: list[str], layer: int, unit: int) -> np.ndarray:
    sae = get_sae(layer)
    tokens = eval_model.to_tokens(texts).to(device)
    logits = eval_model.run_with_hooks(
        tokens,
        return_type="logits",
        fwd_hooks=[(hook_name(layer), partial(hook_ablate_feature, sae=sae, feature_idx=int(unit)))],
    )
    return per_sample_nll(logits, tokens).cpu().numpy()

In [ ]:
rows = []

for (layer, unit), g in prompts.groupby(["layer", "unit"]):
    layer = int(layer)
    unit = int(unit)

    single = g[g["label"] == "single"]["text"].tolist()
    mixed  = g[g["label"] == "mixed"]["text"].tolist()

    #single-concept prompts
    recon_s = losses_recon_only(single, layer)
    ablt_s  = losses_ablate(single, layer, unit)
    delta_s = ablt_s - recon_s
    rel_s   = delta_s / recon_s

    #mixed-concept prompts
    recon_m = losses_recon_only(mixed, layer)
    ablt_m  = losses_ablate(mixed, layer, unit)
    delta_m = ablt_m - recon_m
    rel_m   = delta_m / recon_m

    rows.append({
        "layer": layer,
        "unit": unit,
        "n_single": len(single),
        "n_mixed": len(mixed),

        "delta_feature_single_mean": float(delta_s.mean()),
        "delta_feature_mixed_mean": float(delta_m.mean()),

        "rel_feature_single_mean": float(rel_s.mean()),
        "rel_feature_mixed_mean": float(rel_m.mean()),

        #Interference penalty: single - mixed
        "interference_penalty": float(rel_s.mean() - rel_m.mean()),
    })

out = pd.DataFrame(rows)

out = out.merge(metrics, on=["layer", "unit"], how="left")


In [ ]:
from scipy.stats import spearmanr

#Base filter: active features
df = out[out["max_mad_x"] > 0].copy()

#PRISM polysemanticity score
df["poly_score"] = (
    df["cosine_similarity_random_x"] - df["cosine_similarity_x"]
)

#Polysemanticity threshold
df = df[df["poly_score"] > 0.1]

#stats
x = df["interference"].dropna()

print("n:", len(x))
print("mean:", x.mean())
print("median:", x.median())
print("frac_pos:", (x > 0).mean())

#correlation
y = df[["poly_score", "interference"]].dropna()

if len(y) >= 3:
    r, p = spearmanr(y["poly_score"], y["interference"])
    print("spearman r:", r)
    print("p-value:", p)
else:
    print("not enough points for correlation")

In [ ]:
#Save dataframe with runtime interference units for model filtering
units_all = df[["layer", "unit"]].drop_duplicates()
df_pos = df[df["interference_penalty"] > 0]
units_pos_gpt2_layer0 = df_pos[["layer", "unit"]].drop_duplicates()

In [ ]:
df_all = pd.concat([units_pos_gpt2_layer0,units_pos_gpt2_layer5,units_pos_gpt2_layer10,units_pos_gemma_layer0,units_pos_gemma_layer10,units_pos_gemma_layer20], ignore_index=True)
df_all.to_csv("runtime_interference_units.csv")